In [1]:
###############################################################
# Librerías necesarias
###############################################################

using CSV
using DataFrames
using Downloads
using Statistics
using Random
using MLJ
using GLMNet
using DecisionTree
using LinearAlgebra

###############################################################
# 1. Descargar y cargar datos
###############################################################

url = "https://raw.githubusercontent.com/VC2015/DMLonGitHub/master/penn_jae.dat"
destfile = "penn_jae.dat"

Downloads.download(url, destfile)

df = CSV.read(destfile, DataFrame, delim=' ', ignorerepeated=true)

println("✔ Datos cargados correctamente")
println("Tamaño: ", size(df))
first(df, 5) |> display




✔ Datos cargados correctamente
Tamaño: (13913, 23)


Row,abdt,tg,inuidur1,inuidur2,female,black,hispanic,othrace,dep,q1,q2,q3,q4,q5,q6,recall,agelt35,agegt54,durable,nondurable,lusd,husd,muld
,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64
1,10824,0,18,18,0,0,0,0,2,0,0,0,0,1,0,0,0,0,0,0,0,1,0
2,10635,2,7,3,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0
3,10551,5,18,6,1,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0
4,10824,0,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0
5,10747,0,27,27,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0


In [2]:
###############################################################
# 2. Filtrar tg = 0 o 4
###############################################################

df = df[(df.tg .== 0) .| (df.tg .== 4), :]

###############################################################
# 3. Variable de tratamiento
###############################################################
df.T4 = Int.(df.tg .== 4)


###############################################################
# 4. Variable de resultado
###############################################################
df.y = log.(df.inuidur1)


###############################################################
# 5. Crear dummies para dep (FORMA CORRECTA)
###############################################################

# Crear etiqueta dep_dummy (una fila por observación)
df.dep_1 = Int.(df.dep .== 1)
df.dep_2 = Int.(df.dep .== 2)


###############################################################
# 6. Selección de covariables X
###############################################################

x_vars = select(df,
    :female, :black, :othrace,
    :dep_1, :dep_2,
    :q2, :q3, :q4, :q5, :q6,
    :recall, :agelt35, :agegt54,
    :durable, :nondurable, :lusd, :husd
)

# Verificación correcta (solo nombres)
missing = setdiff(names(x_vars), names(df))
if !isempty(missing)
    println("⚠ Variables faltantes: ", missing)
end

# Crear y, d y X
y = df.y
d = df.T4
X = Matrix(x_vars)

println("\n✔ Preparación completa")
println("y length: ", length(y))
println("d length: ", length(d))
println("X size: ", size(X))



✔ Preparación completa
y length: 5099
d length: 5099
X size: (5099, 17)


In [3]:
###############################################################
# 7. Función DML en Julia 
###############################################################

using Flux

function get_model(method)

    if method == "ols"

        fit = (X,y) -> X \ y
        pred = (model, X) -> X * model


    elseif method == "lasso"

        fit = (X, y) -> glmnet(X, y, alpha = 1)

        pred = (model, Xnew) -> begin
            λ = model.lambda[1]
            idx = findfirst(==(λ), model.lambda)

            β0 = model.a0[idx]
            β  = model.betas[:, idx]

            return β0 .+ Xnew * β
        end


    elseif method == "rf"

        fit = (X, y) -> begin
            p = size(X, 2)
            n_subfeatures = max(1, round(Int, sqrt(p)))
            n_trees = 200

            partial_sampling = 0.7
            max_depth = 10
            min_samples_leaf = 1
            min_samples_split = 2
            min_purity_increase = 0.0

            return build_forest(
                y, X,
                n_subfeatures,
                n_trees,
                partial_sampling,
                max_depth,
                min_samples_leaf,
                min_samples_split,
                min_purity_increase
            )
        end

        pred = (model, Xnew) -> apply_forest(model, Xnew)


    elseif method == "nn"

        fit = (X, y) -> begin
            # Convertir a Float32 y normalizar
            Xf = Float32.(X)
            μ = mean(Xf; dims=1)
            σ = std(Xf; dims=1) .+ 1f-6
            Xnorm = (Xf .- μ) ./ σ

            yf = reshape(Float32.(y), 1, :)

            model = Chain(
                Dense(size(X,2), 16, relu),
                Dense(16, 8, relu),
                Dense(8, 1)
            )

            opt = ADAM()
            state = Flux.setup(opt, model)

            data = [(Xnorm', yf)]

            for epoch in 1:200
                for (xb, yb) in data
                    gs = gradient(model) do m
                        yhat = m(xb)
                        Flux.Losses.mse(yhat, yb)
                    end
                    Flux.update!(state, model, gs)
                end
            end

            return (model=model, μ=μ, σ=σ)  # guardamos normalización
        end

        pred = (pack, Xnew) -> begin
            model = pack.model
            μ = pack.μ
            σ = pack.σ

            Xf = Float32.(Xnew)
            Xnorm = (Xf .- μ) ./ σ

            vec(model(Xnorm'))
        end

    end  # ← ESTE end FALTABA

    return fit, pred
end


get_model (generic function with 1 method)

In [4]:
function dml(y, d, X; method="lasso", n_splits=2, seed=42)

    Random.seed!(seed)
    n = length(y)

    folds = repeat(1:n_splits, inner=ceil(Int, n/n_splits))[1:n]

    ytilde = similar(y, Float64)
    dtilde = similar(d, Float64)

    fit_y, pred_y = get_model(method)
    fit_d, pred_d = get_model(method)

    for k in 1:n_splits
        idx_test = findall(folds .== k)
        idx_train = setdiff(1:n, idx_test)

        # Conversión a Float64 para glmnet y Flux
        Xtrain = Matrix{Float64}(X[idx_train, :])
        Xtest  = Matrix{Float64}(X[idx_test, :])

        ytrain = Float64.(y[idx_train])
        dtrain = Float64.(d[idx_train])

        fy = fit_y(Xtrain, ytrain)
        fd = fit_d(Xtrain, dtrain)

        yhat = pred_y(fy, Xtest)
        dhat = pred_d(fd, Xtest)

        ytilde[idx_test] = y[idx_test] .- yhat
        dtilde[idx_test] = d[idx_test] .- dhat
    end

    theta = sum(dtilde .* ytilde) / sum(dtilde .^ 2)

    resid = ytilde .- theta .* dtilde
    sigma2 = mean(resid .^ 2)
    se = sqrt(sigma2 / sum(dtilde .^ 2))

    return theta, se
end

dml (generic function with 1 method)

In [5]:


###############################################################
# 8. Estimar modelos
###############################################################

modelos = ["ols","lasso","rf","nn"]

results = DataFrame(method=String[], theta=Float64[], se=Float64[], t_stat=Float64[])

for m in modelos
    println("Estimando ", m)
    theta, se = dml(y, d, X; method=m)
    push!(results, (m, theta, se, theta/se))
end

println("Resultados finales:")
results
 

Estimando ols
Estimando lasso
Estimando rf
Estimando nn


┌ Warning: explicit `update!(opt, model, grad)` wants the gradient for the model alone,
│ not the whole tuple from `gradient(m -> loss(m, x, y), model)`. You probably want `grads[1]`.
└ @ Flux C:\Users\FERNANDO\.julia\packages\Flux\uRn8o\src\layers\basic.jl:87
┌ Warning: explicit `update!(opt, model, grad)` wants the gradient for the model alone,
│ not the whole tuple from `gradient(m -> loss(m, x, y), model)`. You probably want `grads[1]`.
└ @ Flux C:\Users\FERNANDO\.julia\packages\Flux\uRn8o\src\layers\basic.jl:87
┌ Warning: explicit `update!(opt, model, grad)` wants the gradient for the model alone,
│ not the whole tuple from `gradient(m -> loss(m, x, y), model)`. You probably want `grads[1]`.
└ @ Flux C:\Users\FERNANDO\.julia\packages\Flux\uRn8o\src\layers\basic.jl:87
┌ Warning: explicit `update!(opt, model, grad)` wants the gradient for the model alone,
│ not the whole tuple from `gradient(m -> loss(m, x, y), model)`. You probably want `grads[1]`.
└ @ Flux C:\Users\FERNANDO\.julia

Resultados finales:


┌ Warning: explicit `update!(opt, model, grad)` wants the gradient for the model alone,
│ not the whole tuple from `gradient(m -> loss(m, x, y), model)`. You probably want `grads[1]`.
└ @ Flux C:\Users\FERNANDO\.julia\packages\Flux\uRn8o\src\layers\basic.jl:87
┌ Warning: explicit `update!(opt, model, grad)` wants the gradient for the model alone,
│ not the whole tuple from `gradient(m -> loss(m, x, y), model)`. You probably want `grads[1]`.
└ @ Flux C:\Users\FERNANDO\.julia\packages\Flux\uRn8o\src\layers\basic.jl:87
┌ Warning: explicit `update!(opt, model, grad)` wants the gradient for the model alone,
│ not the whole tuple from `gradient(m -> loss(m, x, y), model)`. You probably want `grads[1]`.
└ @ Flux C:\Users\FERNANDO\.julia\packages\Flux\uRn8o\src\layers\basic.jl:87


Row,method,theta,se,t_stat
,String,Float64,Float64,Float64
1,ols,-0.0295387,0.0356785,-0.827914
2,lasso,-0.084488,0.0358613,-2.35597
3,rf,-0.0582813,0.0348654,-1.67161
4,nn,-0.0908266,0.0355525,-2.55471


In [6]:
# part 3 DML no CF
function dml_nocf(y, d, X; method="lasso", seed=42)

    Random.seed!(seed)

    n = length(y)

    # crear los vectores residuales
    ytilde = similar(y, Float64)
    dtilde = similar(d, Float64)

    fit_y, pred_y = get_model(method)
    fit_d, pred_d = get_model(method)

    # conversión para glmnet / flux
    Xmat = Matrix{Float64}(X)
    yvec = Float64.(y)
    dvec = Float64.(d)

    # --- entrenar una sola vez ---
    fy = fit_y(Xmat, yvec)
    fd = fit_d(Xmat, dvec)

    # --- predecir usando el mismo conjunto (NO cross-fitting) ---
    yhat = pred_y(fy, Xmat)
    dhat = pred_d(fd, Xmat)

    # --- residuales ---
    ytilde .= yvec .- yhat
    dtilde .= dvec .- dhat

    # --- estimador DML sin CF ---
    theta = sum(dtilde .* ytilde) / sum(dtilde .^ 2)

    resid = ytilde .- theta .* dtilde
    sigma2 = mean(resid .^ 2)
    se = sqrt(sigma2 / sum(dtilde .^ 2))

    return theta, se
end


dml_nocf (generic function with 1 method)

In [7]:
###############################################################
# 9. Estimar modelos SIN CROSS-FITTING
###############################################################

results_nocf = DataFrame(method=String[], theta=Float64[], se=Float64[], t_stat=Float64[])

for m in modelos
    println("Estimando ", m, " (sin cross-fitting)")
    theta, se = dml_nocf(y, d, X; method=m)
    push!(results_nocf, (m, theta, se, theta/se))
end

println("Resultados finales SIN cross-fitting:")
results_nocf


Estimando ols (sin cross-fitting)
Estimando lasso (sin cross-fitting)
Estimando rf (sin cross-fitting)


┌ Warning: explicit `update!(opt, model, grad)` wants the gradient for the model alone,
│ not the whole tuple from `gradient(m -> loss(m, x, y), model)`. You probably want `grads[1]`.
└ @ Flux C:\Users\FERNANDO\.julia\packages\Flux\uRn8o\src\layers\basic.jl:87
┌ Warning: explicit `update!(opt, model, grad)` wants the gradient for the model alone,
│ not the whole tuple from `gradient(m -> loss(m, x, y), model)`. You probably want `grads[1]`.
└ @ Flux C:\Users\FERNANDO\.julia\packages\Flux\uRn8o\src\layers\basic.jl:87
┌ Warning: explicit `update!(opt, model, grad)` wants the gradient for the model alone,
│ not the whole tuple from `gradient(m -> loss(m, x, y), model)`. You probably want `grads[1]`.
└ @ Flux C:\Users\FERNANDO\.julia\packages\Flux\uRn8o\src\layers\basic.jl:87
┌ Warning: explicit `update!(opt, model, grad)` wants the gradient for the model alone,
│ not the whole tuple from `gradient(m -> loss(m, x, y), model)`. You probably want `grads[1]`.
└ @ Flux C:\Users\FERNANDO\.julia

Estimando nn (sin cross-fitting)


┌ Warning: explicit `update!(opt, model, grad)` wants the gradient for the model alone,
│ not the whole tuple from `gradient(m -> loss(m, x, y), model)`. You probably want `grads[1]`.
└ @ Flux C:\Users\FERNANDO\.julia\packages\Flux\uRn8o\src\layers\basic.jl:87
┌ Warning: explicit `update!(opt, model, grad)` wants the gradient for the model alone,
│ not the whole tuple from `gradient(m -> loss(m, x, y), model)`. You probably want `grads[1]`.
└ @ Flux C:\Users\FERNANDO\.julia\packages\Flux\uRn8o\src\layers\basic.jl:87
┌ Warning: explicit `update!(opt, model, grad)` wants the gradient for the model alone,
│ not the whole tuple from `gradient(m -> loss(m, x, y), model)`. You probably want `grads[1]`.
└ @ Flux C:\Users\FERNANDO\.julia\packages\Flux\uRn8o\src\layers\basic.jl:87
┌ Warning: explicit `update!(opt, model, grad)` wants the gradient for the model alone,
│ not the whole tuple from `gradient(m -> loss(m, x, y), model)`. You probably want `grads[1]`.
└ @ Flux C:\Users\FERNANDO\.julia

Resultados finales SIN cross-fitting:


┌ Warning: explicit `update!(opt, model, grad)` wants the gradient for the model alone,
│ not the whole tuple from `gradient(m -> loss(m, x, y), model)`. You probably want `grads[1]`.
└ @ Flux C:\Users\FERNANDO\.julia\packages\Flux\uRn8o\src\layers\basic.jl:87
┌ Warning: explicit `update!(opt, model, grad)` wants the gradient for the model alone,
│ not the whole tuple from `gradient(m -> loss(m, x, y), model)`. You probably want `grads[1]`.
└ @ Flux C:\Users\FERNANDO\.julia\packages\Flux\uRn8o\src\layers\basic.jl:87
┌ Warning: explicit `update!(opt, model, grad)` wants the gradient for the model alone,
│ not the whole tuple from `gradient(m -> loss(m, x, y), model)`. You probably want `grads[1]`.
└ @ Flux C:\Users\FERNANDO\.julia\packages\Flux\uRn8o\src\layers\basic.jl:87
┌ Warning: explicit `update!(opt, model, grad)` wants the gradient for the model alone,
│ not the whole tuple from `gradient(m -> loss(m, x, y), model)`. You probably want `grads[1]`.
└ @ Flux C:\Users\FERNANDO\.julia

Row,method,theta,se,t_stat
,String,Float64,Float64,Float64
1,ols,-0.0338724,0.0357652,-0.947077
2,lasso,-0.0854554,0.0358316,-2.38492
3,rf,-0.0668092,0.0350881,-1.90404
4,nn,-0.062233,0.0364301,-1.70828


2. Principales diferencias
a) Cambios en θ

Los valores de θ cambian al remover cross-fitting, sobre todo en modelos complejos (RF y NN).
Esto refleja sobreajuste en la etapa de predicción cuando el modelo usa los mismos datos para entrenar y predecir.

b) Desviaciones estándar similares

Aunque los SE son similares, esto no significa que el estimador sin cross-fitting sea válido.
El problema central es el sesgo, no la varianza.

c) Cambios en la significancia (t-stat)

Las estadísticas t cambian notablemente (sobre todo en RF y NN), lo que afecta la inferencia.
Sin cross-fitting puede aparentar mayor o menor significancia de manera espuria.

3. Respuestas a las preguntas teóricas
1. ¿Qué se puede decir sobre el RMSE al predecir 
𝑦
y y 
𝑑
d?

El RMSE es más bajo sin cross-fitting.
Esto ocurre porque las predicciones se hacen sobre los mismos datos usados para entrenar, lo cual reduce artificialmente el error (overfitting).

Con cross-fitting, el RMSE es mayor porque se obtiene fuera de muestra, reflejando el verdadero error predictivo.

2. ¿Por qué un método obtiene menor RMSE que el otro?

Porque sin cross-fitting:

el modelo ajusta directamente a todos los datos,

predice sobre las mismas observaciones usadas en el entrenamiento,

y produce residuales más pequeños de lo que deberían ser.

El menor RMSE no indica un modelo mejor, sino un ajuste excesivo.

3. ¿Qué problema tendríamos al estimar sin cross-fitting?

Sin cross-fitting:

Las predicciones 
𝑦
^
y
^
	​

 y 
𝑑
^
d
^
 no son out-of-sample.

Los residuales 
𝑦
~
y
~
	​

 y 
𝑑
~
d
~
 quedan correlacionados con los errores estructurales.

Se viola el supuesto central del DML:

𝐸
[
𝑑
~
 
𝜂
]
=
0
E[
d
~
η]=0

El estimador del efecto causal 
𝜃
θ se vuelve sesgado.

La inferencia deja de ser válida (t-stats engañosos).

Conclusión:
Sin cross-fitting ya no estamos aplicando DML; estamos usando un método parcializado sesgado e inconsistente.